In [ ]:
# Set ENV 
!pip uninstall -y datasets ragas pyarrow
!pip install --no-cache-dir datasets ragas pyarrow requests pandas
!pip install -U langchain-ollama

# ollama pull nomic-embed-text
!ollama pull nomic-embed-text

# 2. Make sure the main LLM model is also downloaded
!ollama pull llama3

In [ ]:
# 1. Install necessary libraries quietly
!pip install ragas langchain-ollama openai datasets pandas -q

import subprocess
import time
import os

print("📥 1. Installing Ollama on the Colab Linux backend...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)

print("🚀 2. Launching Ollama Server in the background...")
# We use subprocess.Popen so the server runs asynchronously without blocking the notebook cell
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for the local server port to open

print("🦙 3. Pulling llama3 model (Loading directly into VRAM)...")
subprocess.run(["ollama", "pull", "llama3"])

print("🧬 4. Pulling nomic-embed-text embedding model...")
subprocess.run(["ollama", "pull", "nomic-embed-text"])

print("✅ Ollama infrastructure setup complete!")

In [ ]:
import os
import ast
import re
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings

# 1. Inject placeholder key to bypass strict OpenAI client checks
os.environ["OPENAI_API_KEY"] = "ollama-local-bypass"

OLLAMA_MODEL_NAME = "llama3" 
OLLAMA_EMBED_NAME = "nomic-embed-text" 

# 2. Connect to the local Ollama API endpoints
ollama_client = OpenAI(
    api_key="ollama", 
    base_url="http://localhost:11434/v1"
)
evaluator_llm = llm_factory(OLLAMA_MODEL_NAME, provider="openai", client=ollama_client)

ollama_embed = OllamaEmbeddings(model=OLLAMA_EMBED_NAME, base_url="http://localhost:11434")
evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)

# 3. Configure metrics with explicit local model wrappers
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextRecall(llm=evaluator_llm)
]

# 4. Fail-safe function to fix malformed stringified lists inside the CSV
CHECKPOINT_FILE = 'progress_checkpoint.csv'

def clean_and_parse_contexts(val):
    if not val or pd.isna(val): 
        return []
    if isinstance(val, list): 
        return val
    val_str = str(val).strip()
    try:
        parsed = ast.literal_eval(val_str)
        if isinstance(parsed, list): 
            return [str(i) for i in parsed]
    except: 
        pass
    matches = re.findall(r"'(.*?)'", val_str)
    if matches: 
        return matches
    return [val_str]

import pandas as pd
from datasets import Dataset

CHECKPOINT_FILE = 'progress_checkpoint.csv'
df_progress = pd.read_csv(CHECKPOINT_FILE)
df_progress['contexts'] = df_progress['contexts'].apply(clean_and_parse_contexts)
df_eval = df_progress[["question", "answer", "contexts", "ground_truth"]]

dataset = Dataset.from_pandas(df_eval)

print("✅ CSV IMPORT !")

# 5. Load and process dataset
if os.path.exists(CHECKPOINT_FILE):
    df_progress = pd.read_csv(CHECKPOINT_FILE)
    df_progress['contexts'] = df_progress['contexts'].apply(clean_and_parse_contexts)
    evaluation_data = df_progress.to_dict(orient='records')
    print(f"🔄 Successfully loaded {len(evaluation_data)} evaluation records from storage.")
else:
    evaluation_data = []
    print("🚨 File Error: Please verify that 'progress_checkpoint.csv' is uploaded to the left sidebar.")

# 6. Execute RAGAS evaluation with asynchronous optimization
if evaluation_data:
    df_eval = pd.DataFrame(evaluation_data)[["question", "answer", "contexts", "ground_truth"]]
    dataset = Dataset.from_pandas(df_eval)

    print("\n📊 Starting RAGAS Evaluation on Colab T4 GPU...")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=metrics,
            llm=evaluator_llm,
            embeddings=evaluator_embeddings,
            is_async=True,  # Enables parallel asynchronous API calls
            batch_size=4    # Safe batch allocation size optimized for T4 VRAM
        )

        print("\n=== ✨ Final Evaluation Summary ===")
        print(result)

        # Export result to CSV
        df_result = result.to_pandas()
        df_result.to_csv('rag_evaluation_report.csv', index=False)
        print("\n✅ Process finished! 'rag_evaluation_report.csv' has been generated.")
        print("💡 Click the 'Refresh' button in the file explorer tab on the left to download your report!")
        
    except Exception as e:
        print(f"🚨 An error occurred during evaluation execution: {e}")

In [ ]:
import json
import requests
import pandas as pd
import os
import ast
from datasets import Dataset
from ragas import evaluate

# Import Latest RAGAS Metrics Classes 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall

# 2. Inject Dummy OpenAI Key to Bypass Framework Check
os.environ["OPENAI_API_KEY"] = "ollama-local-bypass"

# 3. Latest Ollama Native Integration & Metrics Setup (CRITICAL FIX)
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_ollama import OllamaEmbeddings  # Requires: pip install langchain-ollama

# Specify the exact model names you have downloaded in Ollama
OLLAMA_MODEL_NAME = "llama3" 
OLLAMA_EMBED_NAME = "nomic-embed-text"

# Setup local Ollama client via OpenAI v1 compatibility endpoint
ollama_client = OpenAI(
    api_key="ollama", 
    base_url="http://localhost:11434/v1"
)
evaluator_llm = llm_factory(OLLAMA_MODEL_NAME, provider="openai", client=ollama_client)

# Wrap the Ollama Embedding model for Ragas compatibility
ollama_embed = OllamaEmbeddings(model=OLLAMA_EMBED_NAME, base_url="http://localhost:11434")
evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)

# 💡 [CRITICAL FIX] Explicitly inject your local LLM and Embeddings into each metric.
# This prevents Ragas from making stealth OpenAI API calls or throwing structure validation errors.
metrics = [
    Faithfulness(llm=evaluator_llm),
    AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    ContextRecall(llm=evaluator_llm)
]

print(f"🤖 Ollama Model Setup Complete! (LLM: {OLLAMA_MODEL_NAME}, Embed: {OLLAMA_EMBED_NAME})")

# Backend API 
API_URL = "http://localhost:8000/ask"
CHECKPOINT_FILE = 'progress_checkpoint.csv'

# 1. Load Golden Set
if os.path.exists('golden_set.json'):
    with open('golden_set.json', 'r', encoding='utf-8') as f:
        golden_set = json.load(f)
else:
    golden_set = []

# 2. Check for intermediate progress checkpoint (Resume logic)
if os.path.exists(CHECKPOINT_FILE):
    df_progress = pd.read_csv(CHECKPOINT_FILE)
    evaluation_data = df_progress.to_dict(orient='records')
    for item in evaluation_data:
        if isinstance(item['contexts'], str):
            try:
                item['contexts'] = ast.literal_eval(item['contexts'])
            except:
                item['contexts'] = [item['contexts']]
                
    completed_ids = set(df_progress['id'].tolist())
    print(f"🔄 Checkpoint found! Skipping the {len(completed_ids)} already processed items and resuming...")
else:
    evaluation_data = []
    completed_ids = set()

print("🚀 Starting RAG Benchmark Evaluation...")

# 3. Iterate through 50 questions (Will be skipped since all 50 are already collected)
for item in golden_set:
    q_id = item['id']
    query = item['question']
    
    if q_id in completed_ids:
        continue
        
    print(f"Testing {q_id}: {query[:40]}...")
    
    try:
        response = requests.post(API_URL, json={"query": query}, timeout=120)
        if response.status_code == 200:
            data = response.json()
            retrieved_contexts = data.get('sources', [])
            chatbot_answer = data.get('answer', '')
            
            new_row = {
                "id": q_id,
                "question": query,
                "answer": chatbot_answer,
                "contexts": retrieved_contexts,
                "ground_truth": item['ground_truth']
            }
            evaluation_data.append(new_row)
            completed_ids.add(q_id)
            
            df_temp = pd.DataFrame(evaluation_data)
            df_temp.to_csv(CHECKPOINT_FILE, index=False)
        else:
            print(f"❌ Error at {q_id}: Status Code {response.status_code}.")
    except Exception as e:
        print(f"💥 Request failed at {q_id}: {e}.")

# 4. Prepare RAGAS Evaluation
if not evaluation_data:
    print("🚨 No evaluation data collected.")
else:
    df_eval = pd.DataFrame(evaluation_data)[["question", "answer", "contexts", "ground_truth"]]
    dataset = Dataset.from_pandas(df_eval)

    # 5. Run RAGAS Evaluation
    print(f"\n📊 Starting RAGAS Scores calculation for {len(df_eval)} items using Ollama (This might take a few minutes)...")
    try:
        result = evaluate(
            dataset=dataset,
            metrics=metrics,
            llm=evaluator_llm,
            embeddings=evaluator_embeddings
        )

        # 6. Output and save final report
        print("\n=== Final Evaluation Results ===")
        print(result)

        df_result = result.to_pandas()
        df_result.to_csv('rag_evaluation_report.csv', index=False)
        print("✅ Evaluation complete! Report saved to 'rag_evaluation_report.csv'")
        
        # Clean up the checkpoint file once successfully completed
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
            
    except Exception as eval_err:
        print(f"🚨 An error occurred during RAGAS evaluation: {eval_err}")
    

/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_4644/3372156914.py:10: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_4644/3372156914.py:10: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextRecall
/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_4644/3372156914.py:10: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ra

🤖 Ollama Model Setup Complete! (LLM: llama3, Embed: nomic-embed-text)
🔄 Checkpoint found! Skipping the 50 already processed items and resuming...
🚀 Starting RAG Benchmark Evaluation...

📊 Starting RAGAS Scores calculation for 50 items using Ollama (This might take a few minutes)...


/var/folders/b3/4qrylwt92hz6d_yvrfn7fhxh0000gn/T/ipykernel_4644/3372156914.py:34: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(ollama_embed)


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


KeyboardInterrupt: 

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
